In [2]:
from pathlib import Path

code = r'''from pathlib import Path
import math
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# =============================================================================
# CONFIGURATION — edit only these paths if needed
# =============================================================================

PRE_ADJUDICATION_FILE = Path("binary_questions_merged_not_yet_adjudicated.xlsx")
ADJUDICATED_FILE = Path("binary_questions_adjudicated.xlsx")

OUTPUT_XLSX = Path("binary_validation_statistics.xlsx")
OUTPUT_MD = Path("binary_validation_report.md")

N_BOOTSTRAP = 2000       # Use 5000 for final paper numbers if desired
RANDOM_SEED = 20260812
VALID_ANSWERS = {"A", "B"}

ADJUDICATION_COLUMNS = [
    "adjudicated_by",
    "dispute_type",
    "final_answer",
    "status",
    "adjudication_comment",
]

# =============================================================================
# NORMALIZATION / BASIC HELPERS
# =============================================================================

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[_/–—-]+", " ", text)
    text = re.sub(r"[’'`]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_answer(value):
    text = normalize_text(value).upper()
    aliases = {
        "A": "A",
        "B": "B",
        "RESPONSE A": "A",
        "RESPONSE B": "B",
        "REPONSE A": "A",
        "REPONSE B": "B",
        "OPTION A": "A",
        "OPTION B": "B",
    }
    if text in aliases:
        return aliases[text]
    m = re.fullmatch(r"(?:RESPONSE|REPONSE|OPTION)?\s*([AB])", text)
    return m.group(1) if m else ""


def normalize_status(value):
    """
    Robustly normalizes common English/French variants.
    Returns: accepted / revised / rejected / ""
    """
    t = normalize_text(value)
    aliases = {
        # accepted
        "accepted": "accepted",
        "accept": "accepted",
        "accepte": "accepted",
        "acceptee": "accepted",
        "valide": "accepted",
        "validee": "accepted",
        "kept": "accepted",
        "retain": "accepted",
        "retained": "accepted",

        # revised
        "revised": "revised",
        "revise": "revised",
        "revisee": "revised",
        "corrected": "revised",
        "corrige": "revised",
        "corrigee": "revised",
        "modified": "revised",
        "modifie": "revised",
        "modifiee": "revised",

        # rejected
        "rejected": "rejected",
        "reject": "rejected",
        "rejete": "rejected",
        "rejetee": "rejected",
        "discarded": "rejected",
        "exclude": "rejected",
        "excluded": "rejected",
        "exclu": "rejected",
        "exclue": "rejected",
    }
    return aliases.get(t, "")


def canonical_cell(value):
    """For audit comparisons across two Excel reads."""
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        if math.isnan(float(value)):
            return ""
        if float(value).is_integer():
            return str(int(value))
        return f"{float(value):.12g}"
    return re.sub(r"\s+", " ", str(value).strip())


def safe_ratio(num, den):
    return np.nan if den == 0 else float(num) / float(den)


def wilson_ci(successes, n, z=1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = z * math.sqrt((p*(1-p)/n) + (z*z/(4*n*n))) / denom
    return max(0.0, center-half), min(1.0, center+half)


def pct(x):
    return "NA" if pd.isna(x) else f"{100*x:.1f}%"


def fmt3(x):
    return "NA" if pd.isna(x) else f"{x:.3f}"


# =============================================================================
# AGREEMENT COEFFICIENTS
# =============================================================================

def _valid_pairs(a, b):
    x = pd.Series(a).map(normalize_answer)
    y = pd.Series(b).map(normalize_answer)
    mask = x.isin(VALID_ANSWERS) & y.isin(VALID_ANSWERS)
    return x[mask].reset_index(drop=True), y[mask].reset_index(drop=True)


def raw_agreement(a, b):
    x, y = _valid_pairs(a, b)
    if len(x) == 0:
        return np.nan, 0
    return float((x == y).mean()), len(x)


def cohen_kappa(a, b):
    x, y = _valid_pairs(a, b)
    n = len(x)
    if n == 0:
        return np.nan
    po = float((x == y).mean())
    px = x.value_counts(normalize=True)
    py = y.value_counts(normalize=True)
    pe = sum(float(px.get(c, 0)) * float(py.get(c, 0)) for c in ["A", "B"])
    if math.isclose(1 - pe, 0.0):
        return np.nan
    return (po - pe) / (1 - pe)


def gwet_ac1(a, b):
    """
    Gwet's AC1 for nominal binary ratings.
    p_c is the pooled marginal proportion across both raters.
    """
    x, y = _valid_pairs(a, b)
    n = len(x)
    if n == 0:
        return np.nan
    po = float((x == y).mean())
    pooled = pd.concat([x, y], ignore_index=True)
    p = {c: float((pooled == c).mean()) for c in ["A", "B"]}
    k = 2
    pe = sum(pc * (1 - pc) for pc in p.values()) / (k - 1)
    if math.isclose(1 - pe, 0.0):
        return np.nan
    return (po - pe) / (1 - pe)


def krippendorff_alpha_nominal_pairs(a, b):
    """
    Krippendorff's alpha (nominal) for items with two ratings each.
    This is appropriate for the rotating validator pool because it does not
    require the same two validators to rate every item.
    """
    x, y = _valid_pairs(a, b)
    n_items = len(x)
    if n_items == 0:
        return np.nan

    do = float((x != y).mean())

    pooled = pd.concat([x, y], ignore_index=True)
    N = len(pooled)
    if N < 2:
        return np.nan

    counts = pooled.value_counts()
    agree_expected = sum(v * (v - 1) for v in counts.values) / (N * (N - 1))
    de = 1.0 - agree_expected

    if math.isclose(de, 0.0):
        return np.nan
    return 1.0 - (do / de)


def fleiss_kappa_three_votes(df):
    """
    Operational 3-vote Fleiss kappa:
    pre-validation key + validator 1 + validator 2.
    These are NOT three independent validators.
    """
    rows = []
    for r in df[["original_key", "annotator_1_answer", "annotator_2_answer"]].itertuples(index=False):
        votes = [normalize_answer(v) for v in r]
        if all(v in VALID_ANSWERS for v in votes):
            rows.append(votes)

    if not rows:
        return np.nan, np.nan, 0

    n_raters = 3
    category_totals = Counter()
    item_pairwise_agreements = []

    for votes in rows:
        counts = Counter(votes)
        category_totals.update(votes)
        p_i = (sum(v*v for v in counts.values()) - n_raters) / (
            n_raters * (n_raters - 1)
        )
        item_pairwise_agreements.append(p_i)

    p_bar = float(np.mean(item_pairwise_agreements))
    total_ratings = len(rows) * n_raters
    p_j = {c: category_totals.get(c, 0) / total_ratings for c in ["A", "B"]}
    p_e = sum(v*v for v in p_j.values())
    kappa = np.nan if math.isclose(1-p_e, 0.0) else (p_bar-p_e)/(1-p_e)
    return p_bar, kappa, len(rows)


# =============================================================================
# BOOTSTRAP
# =============================================================================

def bootstrap_ci_rows(df, metric_func, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    if len(df) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    vals = []
    n = len(df)
    for _ in range(n_boot):
        sample = df.iloc[rng.integers(0, n, size=n)]
        try:
            value = metric_func(sample)
            if value is not None and np.isfinite(value):
                vals.append(float(value))
        except Exception:
            pass
    if len(vals) < max(50, n_boot // 20):
        return (np.nan, np.nan)
    return tuple(np.quantile(vals, [0.025, 0.975]))


# =============================================================================
# INPUT LOADING
# =============================================================================

def read_required_sheet(path, sheet):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    xls = pd.ExcelFile(path)
    if sheet not in xls.sheet_names:
        raise ValueError(f"Sheet {sheet!r} not found in {path}. Found: {xls.sheet_names}")
    df = pd.read_excel(path, sheet_name=sheet, dtype=object)
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed:")]
    df.columns = [str(c).strip() for c in df.columns]
    return df.dropna(how="all").reset_index(drop=True)


def read_optional_sheet(path, sheet):
    xls = pd.ExcelFile(path)
    if sheet not in xls.sheet_names:
        return pd.DataFrame()
    df = pd.read_excel(path, sheet_name=sheet, dtype=object)
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed:")]
    df.columns = [str(c).strip() for c in df.columns]
    return df.dropna(how="all").reset_index(drop=True)


pre_con = read_required_sheet(PRE_ADJUDICATION_FILE, "CONSOLIDATION")
pre_queue = read_required_sheet(PRE_ADJUDICATION_FILE, "ADJUDICATION_QUEUE")
adj_queue = read_required_sheet(ADJUDICATED_FILE, "ADJUDICATION_QUEUE")
long_ann = read_optional_sheet(PRE_ADJUDICATION_FILE, "LONG_ANNOTATIONS")
quality_report = read_optional_sheet(PRE_ADJUDICATION_FILE, "QUALITY_REPORT")

required_con_cols = [
    "item_id", "Axe", "Difficulty", "intent_group", "intent_category",
    "input", "response_A", "response_B",
    "original_key",
    "annotator_1_id", "annotator_1_answer",
    "annotator_2_id", "annotator_2_answer",
    "provisional_answer", "automatic_decision",
]
missing = [c for c in required_con_cols if c not in pre_con.columns]
if missing:
    raise KeyError("Missing columns from CONSOLIDATION: " + ", ".join(missing))

required_queue_cols = ["item_id"] + ADJUDICATION_COLUMNS
missing = [c for c in required_queue_cols if c not in adj_queue.columns]
if missing:
    raise KeyError("Missing columns from adjudicated ADJUDICATION_QUEUE: " + ", ".join(missing))

for df in [pre_con, pre_queue, adj_queue]:
    df["item_id"] = df["item_id"].map(clean_text)

for col in ["original_key", "annotator_1_answer", "annotator_2_answer", "provisional_answer"]:
    if col in pre_con.columns:
        pre_con[col] = pre_con[col].map(normalize_answer)

adj_queue["final_answer_norm"] = adj_queue["final_answer"].map(normalize_answer)
adj_queue["status_norm"] = adj_queue["status"].map(normalize_status)

# =============================================================================
# BASIC INTEGRITY / RECONSTRUCTION
# =============================================================================

if pre_con["item_id"].duplicated().any():
    raise ValueError("Duplicate item_id values in pre-adjudication CONSOLIDATION.")

if pre_queue["item_id"].duplicated().any():
    raise ValueError("Duplicate item_id values in pre-adjudication ADJUDICATION_QUEUE.")

if adj_queue["item_id"].duplicated().any():
    raise ValueError("Duplicate item_id values in adjudicated ADJUDICATION_QUEUE.")

expected_queue_ids = set(
    pre_con.loc[pre_con["automatic_decision"].astype(str).str.strip().ne("accept"), "item_id"]
)
pre_queue_ids = set(pre_queue["item_id"])
adj_queue_ids = set(adj_queue["item_id"])

queue_id_check = expected_queue_ids == pre_queue_ids == adj_queue_ids

audit_shared_cols = [
    c for c in pre_queue.columns
    if c in adj_queue.columns and c not in ADJUDICATION_COLUMNS and c != "item_id"
]

before = pre_queue.set_index("item_id")
after = adj_queue.set_index("item_id")

queue_change_rows = []
for item_id in sorted(pre_queue_ids & adj_queue_ids):
    for col in audit_shared_cols:
        v1 = canonical_cell(before.at[item_id, col])
        v2 = canonical_cell(after.at[item_id, col])
        if v1 != v2:
            queue_change_rows.append({
                "item_id": item_id,
                "column": col,
                "before": v1,
                "after": v2,
            })

queue_change_audit = pd.DataFrame(
    queue_change_rows,
    columns=["item_id", "column", "before", "after"]
)

adj_fields = adj_queue[
    ["item_id", "adjudicated_by", "dispute_type", "final_answer",
     "final_answer_norm", "status", "status_norm", "adjudication_comment"]
].copy()

items = pre_con.merge(adj_fields, on="item_id", how="left", suffixes=("", "_adj"))

is_auto = items["automatic_decision"].astype(str).str.strip().eq("accept")
is_queued = ~is_auto

items["adjudication_complete"] = False
items.loc[is_queued & items["status_norm"].eq("rejected"), "adjudication_complete"] = True
items.loc[
    is_queued
    & items["status_norm"].isin(["accepted", "revised"])
    & items["final_answer_norm"].isin(VALID_ANSWERS),
    "adjudication_complete"
] = True

items["retained_after_adjudication"] = False
items.loc[is_auto, "retained_after_adjudication"] = True
items.loc[
    is_queued
    & items["status_norm"].isin(["accepted", "revised"])
    & items["final_answer_norm"].isin(VALID_ANSWERS),
    "retained_after_adjudication"
] = True

items["resolved_final_answer"] = ""
items.loc[is_auto, "resolved_final_answer"] = items.loc[is_auto, "original_key"]
items.loc[
    is_queued & items["retained_after_adjudication"],
    "resolved_final_answer"
] = items.loc[
    is_queued & items["retained_after_adjudication"],
    "final_answer_norm"
]

items["original_key_changed"] = (
    items["retained_after_adjudication"]
    & items["resolved_final_answer"].isin(VALID_ANSWERS)
    & items["original_key"].isin(VALID_ANSWERS)
    & items["resolved_final_answer"].ne(items["original_key"])
)

items["validators_agree"] = (
    items["annotator_1_answer"].isin(VALID_ANSWERS)
    & items["annotator_2_answer"].isin(VALID_ANSWERS)
    & items["annotator_1_answer"].eq(items["annotator_2_answer"])
)

items["both_validators_against_key"] = (
    items["validators_agree"]
    & items["original_key"].isin(VALID_ANSWERS)
    & items["annotator_1_answer"].ne(items["original_key"])
)

items["validators_disagree"] = (
    items["annotator_1_answer"].isin(VALID_ANSWERS)
    & items["annotator_2_answer"].isin(VALID_ANSWERS)
    & items["annotator_1_answer"].ne(items["annotator_2_answer"])
)

items["three_way_unanimous"] = (
    items["original_key"].isin(VALID_ANSWERS)
    & items["annotator_1_answer"].isin(VALID_ANSWERS)
    & items["annotator_2_answer"].isin(VALID_ANSWERS)
    & items["original_key"].eq(items["annotator_1_answer"])
    & items["original_key"].eq(items["annotator_2_answer"])
)

# =============================================================================
# METRIC TABLE HELPERS
# =============================================================================

metric_rows = []

def add_metric(section, metric, value, n="", ci=(np.nan, np.nan), note=""):
    metric_rows.append({
        "section": section,
        "metric": metric,
        "value": value,
        "n": n,
        "ci95_low": ci[0],
        "ci95_high": ci[1],
        "note": note,
    })


def add_rate_metric(section, metric, successes, n, note=""):
    value = safe_ratio(successes, n)
    ci = wilson_ci(successes, n)
    add_metric(section, metric, value, n, ci, note)


# =============================================================================
# DATASET FLOW / ADJUDICATION
# =============================================================================

N = len(items)
n_auto = int(is_auto.sum())
n_queue = int(is_queued.sum())
n_complete = int(items.loc[is_queued, "adjudication_complete"].sum())

add_metric("dataset_flow", "initial_binary_items", N, N)
add_metric("dataset_flow", "automatic_accept_items", n_auto, N)
add_metric("dataset_flow", "adjudication_queue_items", n_queue, N)
add_rate_metric("dataset_flow", "automatic_accept_rate", n_auto, N)
add_rate_metric("dataset_flow", "adjudication_rate", n_queue, N)

add_rate_metric("adjudication", "adjudication_completion_rate", n_complete, n_queue)

for status in ["accepted", "revised", "rejected"]:
    count = int(items.loc[is_queued, "status_norm"].eq(status).sum())
    add_metric("adjudication", f"status_{status}_count", count, n_queue)
    add_rate_metric(
        "adjudication",
        f"status_{status}_rate_among_queued",
        count,
        n_queue,
    )

n_retained = int(items["retained_after_adjudication"].sum())
if n_complete == n_queue:
    add_metric("dataset_flow", "retained_after_adjudication_items", n_retained, N)
    add_rate_metric("dataset_flow", "post_adjudication_retention_rate", n_retained, N)
else:
    add_metric(
        "dataset_flow",
        "retained_after_adjudication_items_so_far",
        n_retained,
        N,
        note="Adjudication queue is not fully complete; final retention rate is therefore not reported."
    )

retained = items[items["retained_after_adjudication"]].copy()
n_changed = int(retained["original_key_changed"].sum())
add_rate_metric(
    "final_key",
    "original_key_changed_rate_among_retained",
    n_changed,
    len(retained),
)

adj_retained = items[
    is_queued & items["retained_after_adjudication"]
].copy()
add_rate_metric(
    "final_key",
    "original_key_changed_rate_among_adjudicated_retained",
    int(adj_retained["original_key_changed"].sum()),
    len(adj_retained),
)

maj = items[
    is_queued
    & items["adjudication_complete"]
    & items["provisional_answer"].isin(VALID_ANSWERS)
    & items["final_answer_norm"].isin(VALID_ANSWERS)
].copy()
add_rate_metric(
    "adjudication",
    "adjudicator_follows_provisional_majority_rate",
    int(maj["final_answer_norm"].eq(maj["provisional_answer"]).sum()),
    len(maj),
)

both_against = items[
    is_queued
    & items["both_validators_against_key"]
    & items["adjudication_complete"]
    & items["final_answer_norm"].isin(VALID_ANSWERS)
].copy()
add_rate_metric(
    "adjudication",
    "when_both_validators_disagree_with_key_final_follows_validators_rate",
    int(both_against["final_answer_norm"].eq(both_against["annotator_1_answer"]).sum()),
    len(both_against),
    note=(
        "Diagnostic: both independent validators agree with each other and oppose "
        "the initial key; measures how often senior adjudication follows them."
    ),
)

split_disagreement = items[
    is_queued
    & items["validators_disagree"]
    & items["adjudication_complete"]
    & items["final_answer_norm"].isin(VALID_ANSWERS)
].copy()
add_rate_metric(
    "adjudication",
    "when_validators_disagree_final_follows_initial_key_rate",
    int(split_disagreement["final_answer_norm"].eq(split_disagreement["original_key"]).sum()),
    len(split_disagreement),
    note=(
        "With binary answers, validator disagreement implies one validator agrees "
        "with the initial key and one disagrees."
    ),
)

# =============================================================================
# RELIABILITY: TWO INDEPENDENT VALIDATORS
# =============================================================================

pair_df = items[
    items["annotator_1_answer"].isin(VALID_ANSWERS)
    & items["annotator_2_answer"].isin(VALID_ANSWERS)
].copy()

raw_vv, n_vv = raw_agreement(pair_df["annotator_1_answer"], pair_df["annotator_2_answer"])
raw_successes = int(
    pair_df["annotator_1_answer"].eq(pair_df["annotator_2_answer"]).sum()
)
add_metric(
    "reliability",
    "validator1_vs_validator2__raw_agreement",
    raw_vv,
    n_vv,
    wilson_ci(raw_successes, n_vv),
    note="Direct agreement between the two independently assigned validators."
)

for metric_name, func in [
    ("validator1_vs_validator2__cohen_kappa",
     lambda d: cohen_kappa(d["annotator_1_answer"], d["annotator_2_answer"])),
    ("validator1_vs_validator2__gwet_ac1",
     lambda d: gwet_ac1(d["annotator_1_answer"], d["annotator_2_answer"])),
    ("krippendorff_alpha_actual_validator_pool",
     lambda d: krippendorff_alpha_nominal_pairs(
         d["annotator_1_answer"], d["annotator_2_answer"]
     )),
]:
    value = func(pair_df)
    ci = bootstrap_ci_rows(pair_df, func)
    note = ""
    if metric_name == "krippendorff_alpha_actual_validator_pool":
        note = (
            "Primary global IRR statistic. Different pairs from the nine-validator "
            "pool reviewed different items."
        )
    add_metric("reliability", metric_name, value, len(pair_df), ci, note)

# Initial key vs validator judgments (validation consistency, not pure IRR)
key_long = pd.concat([
    items[["item_id", "original_key", "annotator_1_answer"]]
        .rename(columns={"annotator_1_answer": "validator_answer"})
        .assign(slot="validator_1"),
    items[["item_id", "original_key", "annotator_2_answer"]]
        .rename(columns={"annotator_2_answer": "validator_answer"})
        .assign(slot="validator_2"),
], ignore_index=True)

key_long = key_long[
    key_long["original_key"].isin(VALID_ANSWERS)
    & key_long["validator_answer"].isin(VALID_ANSWERS)
].copy()

raw_kv, n_kv = raw_agreement(key_long["original_key"], key_long["validator_answer"])
kv_success = int(key_long["original_key"].eq(key_long["validator_answer"]).sum())
add_metric(
    "key_validation",
    "initial_key_vs_all_validator_judgments__raw_agreement",
    raw_kv,
    n_kv,
    wilson_ci(kv_success, n_kv),
    note=(
        "Descriptive validation agreement. Each item contributes two key-validator "
        "comparisons; this is not the primary inter-rater reliability measure."
    ),
)

def pooled_key_kappa(d):
    a = pd.concat([d["original_key"], d["original_key"]], ignore_index=True)
    b = pd.concat([d["annotator_1_answer"], d["annotator_2_answer"]], ignore_index=True)
    return cohen_kappa(a, b)

def pooled_key_ac1(d):
    a = pd.concat([d["original_key"], d["original_key"]], ignore_index=True)
    b = pd.concat([d["annotator_1_answer"], d["annotator_2_answer"]], ignore_index=True)
    return gwet_ac1(a, b)

for name, func in [
    ("initial_key_vs_all_validator_judgments__cohen_kappa", pooled_key_kappa),
    ("initial_key_vs_all_validator_judgments__gwet_ac1", pooled_key_ac1),
]:
    value = func(items)
    ci = bootstrap_ci_rows(items, func)
    add_metric(
        "key_validation", name, value, len(items), ci,
        note="Supplementary construction-stage key validation statistic."
    )

# =============================================================================
# THREE-WAY OPERATIONAL CONSENSUS
# =============================================================================

complete_three = items[
    items["original_key"].isin(VALID_ANSWERS)
    & items["annotator_1_answer"].isin(VALID_ANSWERS)
    & items["annotator_2_answer"].isin(VALID_ANSWERS)
].copy()

n_three = len(complete_three)
n_unanimous = int(complete_three["three_way_unanimous"].sum())
add_rate_metric(
    "three_way_consensus",
    "three_way_unanimous_rate",
    n_unanimous,
    n_three,
    note="Initial pre-validation key + two independent validators."
)
add_rate_metric(
    "three_way_consensus",
    "three_way_2_of_3_majority_rate",
    n_three - n_unanimous,
    n_three,
    note=(
        "For binary A/B items, every valid non-unanimous three-vote case necessarily "
        "has a 2/3 majority."
    )
)

pbar, fleiss, n_f = fleiss_kappa_three_votes(complete_three)
add_metric(
    "three_way_consensus",
    "three_way_mean_pairwise_agreement",
    pbar,
    n_f,
    note="Operational consensus measure, not three-independent-rater reliability."
)

fleiss_ci = bootstrap_ci_rows(
    complete_three,
    lambda d: fleiss_kappa_three_votes(d)[1]
)
add_metric(
    "three_way_consensus",
    "three_way_fleiss_kappa",
    fleiss,
    n_f,
    fleiss_ci,
    note="Operational consensus measure including the initial pre-validation key."
)

# =============================================================================
# FLAGS / QUALITY
# =============================================================================

if not long_ann.empty:
    for c in ["flag_language", "flag_safety", "flag_item"]:
        if c in long_ann.columns:
            long_ann[c] = long_ann[c].map(normalize_text)

    if "flag_language" in long_ann.columns:
        valid = long_ann["flag_language"].isin(["ok", "probleme"])
        add_rate_metric(
            "annotation_quality",
            "annotation_level_language_flag_rate",
            int(long_ann.loc[valid, "flag_language"].eq("probleme").sum()),
            int(valid.sum()),
        )
    if "flag_safety" in long_ann.columns:
        valid = long_ann["flag_safety"].isin(["ok", "probleme"])
        add_rate_metric(
            "annotation_quality",
            "annotation_level_safety_flag_rate",
            int(long_ann.loc[valid, "flag_safety"].eq("probleme").sum()),
            int(valid.sum()),
        )
    if "flag_item" in long_ann.columns:
        valid = long_ann["flag_item"].ne("")
        add_rate_metric(
            "annotation_quality",
            "annotation_level_item_problem_rate",
            int(long_ann.loc[valid, "flag_item"].ne("ok").sum()),
            int(valid.sum()),
        )

if "mandatory_flag_review" in items.columns:
    flag_mask = items["mandatory_flag_review"].map(normalize_text).isin(["yes", "oui", "true", "1"])
    add_rate_metric(
        "annotation_quality",
        "item_level_mandatory_flag_review_rate",
        int(flag_mask.sum()),
        len(items),
    )

if "language_review_recommended" in items.columns:
    flag_mask = items["language_review_recommended"].map(normalize_text).isin(["yes", "oui", "true", "1"])
    add_rate_metric(
        "annotation_quality",
        "item_level_language_review_rate",
        int(flag_mask.sum()),
        len(items),
    )

# =============================================================================
# GROUP-WISE STATISTICS
# =============================================================================

group_rows = []

def group_summary(d, grouping, value):
    vv = d[
        d["annotator_1_answer"].isin(VALID_ANSWERS)
        & d["annotator_2_answer"].isin(VALID_ANSWERS)
    ].copy()
    raw, n = raw_agreement(vv["annotator_1_answer"], vv["annotator_2_answer"])
    kappa = cohen_kappa(vv["annotator_1_answer"], vv["annotator_2_answer"])
    ac1 = gwet_ac1(vv["annotator_1_answer"], vv["annotator_2_answer"])
    alpha = krippendorff_alpha_nominal_pairs(
        vv["annotator_1_answer"], vv["annotator_2_answer"]
    )
    unanimity = safe_ratio(int(d["three_way_unanimous"].sum()), len(d))
    adjud_rate = safe_ratio(
        int(d["automatic_decision"].astype(str).str.strip().ne("accept").sum()),
        len(d),
    )
    retained_d = d[d["retained_after_adjudication"]]
    key_change = safe_ratio(
        int(retained_d["original_key_changed"].sum()),
        len(retained_d),
    )
    return {
        "grouping": grouping,
        "group": clean_text(value),
        "n_items": len(d),
        "validator_pair_raw_agreement": raw,
        "validator_pair_cohen_kappa": kappa,
        "validator_pair_gwet_ac1": ac1,
        "validator_pool_krippendorff_alpha": alpha,
        "three_way_unanimous_rate": unanimity,
        "adjudication_rate": adjud_rate,
        "retained_items": len(retained_d),
        "original_key_changed_rate_among_retained": key_change,
    }

for grouping in ["Axe", "Difficulty", "intent_group"]:
    if grouping in items.columns:
        for value, d in items.groupby(grouping, dropna=False):
            group_rows.append(group_summary(d, grouping, value))

agreement_by_group = pd.DataFrame(group_rows)

# =============================================================================
# VALIDATOR-PAIR ANALYSIS
# =============================================================================

pair_rows = []
pair_base = pair_df.copy()
pair_base["validator_pair"] = pair_base.apply(
    lambda r: " × ".join(sorted([clean_text(r["annotator_1_id"]), clean_text(r["annotator_2_id"])])),
    axis=1,
)

for pair, d in pair_base.groupby("validator_pair"):
    raw, n = raw_agreement(d["annotator_1_answer"], d["annotator_2_answer"])
    success = int(d["annotator_1_answer"].eq(d["annotator_2_answer"]).sum())
    lo, hi = wilson_ci(success, n)
    pair_rows.append({
        "validator_pair": pair,
        "n_items": n,
        "raw_agreement": raw,
        "raw_agreement_ci95_low": lo,
        "raw_agreement_ci95_high": hi,
        "cohen_kappa": cohen_kappa(d["annotator_1_answer"], d["annotator_2_answer"]),
        "gwet_ac1": gwet_ac1(d["annotator_1_answer"], d["annotator_2_answer"]),
        "krippendorff_alpha_nominal": krippendorff_alpha_nominal_pairs(
            d["annotator_1_answer"], d["annotator_2_answer"]
        ),
    })

validator_pairs = pd.DataFrame(pair_rows).sort_values(
    ["n_items", "validator_pair"], ascending=[False, True]
).reset_index(drop=True)

# =============================================================================
# VALIDATOR-LEVEL SUMMARY
# =============================================================================

if not long_ann.empty and {"item_id", "annotator_id", "annotator_answer"}.issubset(long_ann.columns):
    la = long_ann.copy()
    la["item_id"] = la["item_id"].map(clean_text)
    la["annotator_id"] = la["annotator_id"].map(clean_text)
    la["annotator_answer"] = la["annotator_answer"].map(normalize_answer)

    reference = items[
        ["item_id", "original_key", "resolved_final_answer", "retained_after_adjudication"]
    ]
    la = la.merge(reference, on="item_id", how="left")

    validator_rows = []
    for ann, d in la.groupby("annotator_id"):
        valid_ans = d["annotator_answer"].isin(VALID_ANSWERS)
        key_valid = valid_ans & d["original_key"].isin(VALID_ANSWERS)
        final_valid = (
            valid_ans
            & d["retained_after_adjudication"].fillna(False)
            & d["resolved_final_answer"].isin(VALID_ANSWERS)
        )
        row = {
            "annotator_id": ann,
            "annotations": len(d),
            "valid_answers": int(valid_ans.sum()),
            "answer_A_rate": safe_ratio(int(d.loc[valid_ans, "annotator_answer"].eq("A").sum()), int(valid_ans.sum())),
            "answer_B_rate": safe_ratio(int(d.loc[valid_ans, "annotator_answer"].eq("B").sum()), int(valid_ans.sum())),
            "agreement_with_initial_key": safe_ratio(
                int(d.loc[key_valid, "annotator_answer"].eq(d.loc[key_valid, "original_key"]).sum()),
                int(key_valid.sum()),
            ),
            "agreement_with_resolved_final_answer": safe_ratio(
                int(d.loc[final_valid, "annotator_answer"].eq(d.loc[final_valid, "resolved_final_answer"]).sum()),
                int(final_valid.sum()),
            ),
        }

        for flag_col, out_name in [
            ("flag_language", "language_flag_rate"),
            ("flag_safety", "safety_flag_rate"),
        ]:
            if flag_col in d.columns:
                vals = d[flag_col].map(normalize_text)
                valid = vals.isin(["ok", "probleme"])
                row[out_name] = safe_ratio(int(vals[valid].eq("probleme").sum()), int(valid.sum()))

        if "flag_item" in d.columns:
            vals = d["flag_item"].map(normalize_text)
            valid = vals.ne("")
            row["item_problem_rate"] = safe_ratio(
                int(vals[valid].ne("ok").sum()), int(valid.sum())
            )

        validator_rows.append(row)

    validators = pd.DataFrame(validator_rows).sort_values("annotator_id").reset_index(drop=True)
else:
    parts = []
    for slot in [1, 2]:
        tmp = items[
            ["item_id", "original_key", "resolved_final_answer", "retained_after_adjudication",
             f"annotator_{slot}_id", f"annotator_{slot}_answer"]
        ].copy()
        tmp = tmp.rename(columns={
            f"annotator_{slot}_id": "annotator_id",
            f"annotator_{slot}_answer": "annotator_answer",
        })
        parts.append(tmp)
    la = pd.concat(parts, ignore_index=True)

    validator_rows = []
    for ann, d in la.groupby("annotator_id"):
        valid_ans = d["annotator_answer"].isin(VALID_ANSWERS)
        key_valid = valid_ans & d["original_key"].isin(VALID_ANSWERS)
        final_valid = (
            valid_ans
            & d["retained_after_adjudication"].fillna(False)
            & d["resolved_final_answer"].isin(VALID_ANSWERS)
        )
        validator_rows.append({
            "annotator_id": ann,
            "annotations": len(d),
            "valid_answers": int(valid_ans.sum()),
            "agreement_with_initial_key": safe_ratio(
                int(d.loc[key_valid, "annotator_answer"].eq(d.loc[key_valid, "original_key"]).sum()),
                int(key_valid.sum()),
            ),
            "agreement_with_resolved_final_answer": safe_ratio(
                int(d.loc[final_valid, "annotator_answer"].eq(d.loc[final_valid, "resolved_final_answer"]).sum()),
                int(final_valid.sum()),
            ),
        })
    validators = pd.DataFrame(validator_rows).sort_values("annotator_id").reset_index(drop=True)

# =============================================================================
# ANSWER DISTRIBUTIONS
# =============================================================================

dist_rows = []

def add_distribution(label, series):
    s = pd.Series(series).map(normalize_answer)
    s = s[s.isin(VALID_ANSWERS)]
    n = len(s)
    for answer in ["A", "B"]:
        count = int((s == answer).sum())
        dist_rows.append({
            "source": label,
            "answer": answer,
            "count": count,
            "n_valid": n,
            "rate": safe_ratio(count, n),
        })

add_distribution("initial_key", items["original_key"])
add_distribution("annotator_1", items["annotator_1_answer"])
add_distribution("annotator_2", items["annotator_2_answer"])
add_distribution(
    "validators_combined",
    pd.concat([items["annotator_1_answer"], items["annotator_2_answer"]], ignore_index=True)
)
add_distribution("resolved_final_answer_retained", retained["resolved_final_answer"])

answer_distributions = pd.DataFrame(dist_rows)

# =============================================================================
# CONFUSION MATRICES — long format
# =============================================================================

conf_rows = []

def add_confusion(label, a, b, row_name, col_name):
    x, y = _valid_pairs(a, b)
    for ra in ["A", "B"]:
        for cb in ["A", "B"]:
            conf_rows.append({
                "comparison": label,
                "row_variable": row_name,
                "column_variable": col_name,
                "row_answer": ra,
                "column_answer": cb,
                "count": int(((x == ra) & (y == cb)).sum()),
            })

add_confusion(
    "validator_1_vs_validator_2",
    items["annotator_1_answer"], items["annotator_2_answer"],
    "validator_1", "validator_2"
)
add_confusion(
    "initial_key_vs_validator_1",
    items["original_key"], items["annotator_1_answer"],
    "initial_key", "validator_1"
)
add_confusion(
    "initial_key_vs_validator_2",
    items["original_key"], items["annotator_2_answer"],
    "initial_key", "validator_2"
)
add_confusion(
    "initial_key_vs_resolved_final_retained",
    retained["original_key"], retained["resolved_final_answer"],
    "initial_key", "resolved_final"
)

confusion_matrices = pd.DataFrame(conf_rows)

# =============================================================================
# ADJUDICATION BREAKDOWNS
# =============================================================================

adjudication_rows = []

def frequency_rows(series, variable):
    s = series.map(lambda x: clean_text(x) if clean_text(x) else "(blank)")
    n = len(s)
    vc = s.value_counts(dropna=False)
    for value, count in vc.items():
        adjudication_rows.append({
            "variable": variable,
            "value": value,
            "count": int(count),
            "denominator": n,
            "rate": safe_ratio(int(count), n),
        })

queued_items = items[is_queued].copy()
frequency_rows(queued_items["status_norm"], "normalized_status")
if "dispute_type" in queued_items.columns:
    frequency_rows(queued_items["dispute_type"], "dispute_type")
if "decision_reason" in queued_items.columns:
    frequency_rows(queued_items["decision_reason"], "automatic_decision_reason")
if "automatic_decision" in queued_items.columns:
    frequency_rows(queued_items["automatic_decision"], "automatic_decision")

adjudication_breakdown = pd.DataFrame(adjudication_rows)

# =============================================================================
# INTEGRITY CHECKS
# =============================================================================

integrity_rows = []

def add_check(check, value, passed, details=""):
    integrity_rows.append({
        "check": check,
        "value": value,
        "pass": "yes" if passed else "no",
        "details": details,
    })

add_check(
    "initial_item_ids_unique",
    pre_con["item_id"].nunique(),
    pre_con["item_id"].nunique() == len(pre_con),
)
add_check(
    "queue_ids_match_nonaccepted_items",
    len(adj_queue_ids),
    queue_id_check,
    f"expected={len(expected_queue_ids)}, pre_queue={len(pre_queue_ids)}, adjudicated_queue={len(adj_queue_ids)}",
)
add_check(
    "unexpected_nonadjudication_cell_changes_in_queue",
    len(queue_change_audit),
    len(queue_change_audit) == 0,
)
add_check(
    "unrecognized_adjudication_statuses",
    int((is_queued & items["status_norm"].eq("")).sum()),
    int((is_queued & items["status_norm"].eq("")).sum()) == 0,
)
invalid_retained_final = int(
    (
        is_queued
        & items["status_norm"].isin(["accepted", "revised"])
        & ~items["final_answer_norm"].isin(VALID_ANSWERS)
    ).sum()
)
add_check(
    "accepted_or_revised_items_without_valid_final_answer",
    invalid_retained_final,
    invalid_retained_final == 0,
)
add_check(
    "adjudication_completion",
    f"{n_complete}/{n_queue}",
    n_complete == n_queue,
)
valid_three_count = len(complete_three)
add_check(
    "items_with_three_valid_votes",
    f"{valid_three_count}/{N}",
    valid_three_count == N,
)
add_check(
    "automatic_accept_items_are_three_way_unanimous",
    int((items.loc[is_auto, "three_way_unanimous"] == False).sum()),
    bool(items.loc[is_auto, "three_way_unanimous"].all()) if n_auto else True,
)

if not long_ann.empty and {"item_id", "annotator_id"}.issubset(long_ann.columns):
    counts = long_ann.groupby("item_id")["annotator_id"].nunique()
    bad_two = int((counts.reindex(items["item_id"], fill_value=0) != 2).sum())
    add_check(
        "items_without_exactly_two_distinct_validators_in_long_annotations",
        bad_two,
        bad_two == 0,
    )

integrity = pd.DataFrame(integrity_rows)

# =============================================================================
# ITEM-LEVEL AUDIT
# =============================================================================

audit_cols = [
    "item_id", "Axe", "Difficulty", "intent_group", "intent_category",
    "input", "response_A", "response_B",
    "original_key",
    "annotator_1_id", "annotator_1_answer",
    "annotator_2_id", "annotator_2_answer",
    "validators_agree", "both_validators_against_key", "validators_disagree",
    "three_way_unanimous",
    "vote_distribution", "provisional_answer",
    "automatic_decision", "decision_reason",
    "mandatory_flag_review", "language_review_recommended",
    "adjudicated_by", "dispute_type",
    "final_answer", "status", "status_norm",
    "adjudication_comment",
    "adjudication_complete",
    "retained_after_adjudication",
    "resolved_final_answer",
    "original_key_changed",
]
audit_cols = [c for c in audit_cols if c in items.columns]
item_level_audit = items[audit_cols].copy()

overall_metrics = pd.DataFrame(metric_rows)

# =============================================================================
# README
# =============================================================================

readme = pd.DataFrame([
    {
        "topic": "Primary IRR statistic",
        "description": (
            "Krippendorff's nominal alpha across the rotating validator pool. "
            "Each binary item was independently judged by two validators."
        ),
    },
    {
        "topic": "Supplementary IRR",
        "description": (
            "Raw validator-validator agreement, pooled slot-wise Cohen's kappa, "
            "and Gwet's AC1."
        ),
    },
    {
        "topic": "Three-way consensus",
        "description": (
            "The pre-validation key + validator 1 + validator 2. These are "
            "operational consensus statistics, not reliability among three independent raters."
        ),
    },
    {
        "topic": "Binary-specific property",
        "description": (
            "With three valid A/B votes, every non-unanimous case necessarily has a 2/3 majority."
        ),
    },
    {
        "topic": "Adjudication completion",
        "description": (
            "Rejected items count as completed without requiring final_answer. "
            "Accepted/revised items require a valid final A/B answer."
        ),
    },
    {
        "topic": "Retention",
        "description": (
            "Without a separate final binary workbook, post-adjudication retention is reconstructed "
            "as automatic accepts plus adjudicated items with status accepted/revised and a valid final answer."
        ),
    },
    {
        "topic": "Confidence intervals",
        "description": (
            "Wilson 95% CIs are used for proportions. Cohen's kappa, Gwet's AC1, "
            "Krippendorff's alpha and Fleiss' kappa use item-level percentile bootstrap CIs."
        ),
    },
])

# =============================================================================
# EXCEL EXPORT
# =============================================================================

def excel_safe(df):
    out = df.copy()
    return out.where(pd.notna(out), None)


with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    excel_safe(readme).to_excel(writer, index=False, sheet_name="README")
    excel_safe(overall_metrics).to_excel(writer, index=False, sheet_name="OVERALL_METRICS")
    excel_safe(agreement_by_group).to_excel(writer, index=False, sheet_name="AGREEMENT_BY_GROUP")
    excel_safe(validator_pairs).to_excel(writer, index=False, sheet_name="VALIDATOR_PAIRS")
    excel_safe(validators).to_excel(writer, index=False, sheet_name="VALIDATORS")
    excel_safe(answer_distributions).to_excel(writer, index=False, sheet_name="ANSWER_DISTRIBUTIONS")
    excel_safe(confusion_matrices).to_excel(writer, index=False, sheet_name="CONFUSION_MATRICES")
    excel_safe(adjudication_breakdown).to_excel(writer, index=False, sheet_name="ADJUDICATION")
    excel_safe(integrity).to_excel(writer, index=False, sheet_name="INTEGRITY")
    excel_safe(queue_change_audit).to_excel(writer, index=False, sheet_name="QUEUE_CHANGE_AUDIT")
    if not quality_report.empty:
        excel_safe(quality_report).to_excel(writer, index=False, sheet_name="INPUT_QUALITY_REPORT")
    excel_safe(item_level_audit).to_excel(writer, index=False, sheet_name="ITEM_LEVEL_AUDIT")

wb = load_workbook(OUTPUT_XLSX)
header_fill = PatternFill("solid", fgColor="D9EAF7")

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.sheet_view.showGridLines = False
    if ws.max_row >= 1 and ws.max_column >= 1:
        ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    for col_idx in range(1, ws.max_column + 1):
        header = clean_text(ws.cell(1, col_idx).value)
        if header in {
            "input", "response_A", "response_B", "description",
            "details", "note", "before", "after", "adjudication_comment"
        }:
            width = 48
        elif "comment" in header.lower() or "reason" in header.lower():
            width = 42
        elif "intent" in header.lower() or header in {"Axe", "metric", "check"}:
            width = 30
        else:
            width = min(max(len(header) + 3, 12), 24)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

wb.save(OUTPUT_XLSX)

# =============================================================================
# MARKDOWN PAPER REPORT
# =============================================================================

metric_lookup = dict(zip(overall_metrics["metric"], overall_metrics["value"]))

def m(name):
    return metric_lookup.get(name, np.nan)

def ci_text(name, percent=False):
    row = overall_metrics.loc[overall_metrics["metric"].eq(name)]
    if row.empty:
        return ""
    lo = row.iloc[0]["ci95_low"]
    hi = row.iloc[0]["ci95_high"]
    if pd.isna(lo) or pd.isna(hi):
        return ""
    return f"95% CI [{pct(lo)}, {pct(hi)}]" if percent else f"95% CI [{lo:.3f}, {hi:.3f}]"

status_counts = {
    s: int(items.loc[is_queued, "status_norm"].eq(s).sum())
    for s in ["accepted", "revised", "rejected"]
}

alpha = m("krippendorff_alpha_actual_validator_pool")
vv_raw = m("validator1_vs_validator2__raw_agreement")
vv_kappa = m("validator1_vs_validator2__cohen_kappa")
vv_ac1 = m("validator1_vs_validator2__gwet_ac1")
three_unanim = m("three_way_unanimous_rate")
three_fleiss = m("three_way_fleiss_kappa")
adjud_rate = m("adjudication_rate")
ret_rate = m("post_adjudication_retention_rate")
key_change = m("original_key_changed_rate_among_retained")
key_change_adj = m("original_key_changed_rate_among_adjudicated_retained")
follow_maj = m("adjudicator_follows_provisional_majority_rate")
completion = m("adjudication_completion_rate")

md = []
md.append("# Binary-question cross-validation and adjudication statistics\n")
md.append(
    f"Inputs: `{PRE_ADJUDICATION_FILE.name}`, `{ADJUDICATED_FILE.name}`.\n"
)

md.append("## Dataset flow\n")
md.append(f"- Initial binary items: **{N}**.")
md.append(f"- Automatically accepted items: **{n_auto} ({pct(n_auto/N)})**.")
md.append(f"- Items entering adjudication: **{n_queue} ({pct(adjud_rate)})**.")
md.append(f"- Adjudication completion rate: **{pct(completion)}**.")
if not pd.isna(ret_rate):
    md.append(f"- Post-adjudication retained items: **{n_retained} ({pct(ret_rate)})**.")
md.append(
    f"- Adjudication statuses: **{status_counts['accepted']} accepted**, "
    f"**{status_counts['revised']} revised**, **{status_counts['rejected']} rejected**."
)

md.append("\n## Inter-validator reliability\n")
md.append(
    f"- Two-validator raw agreement: **{pct(vv_raw)}** "
    f"({ci_text('validator1_vs_validator2__raw_agreement', percent=True)})."
)
md.append(
    f"- Krippendorff's nominal α across the rotating validator pool: "
    f"**{fmt3(alpha)}** ({ci_text('krippendorff_alpha_actual_validator_pool')})."
)
md.append(
    f"- Supplementary pooled diagnostics: Cohen's κ **{fmt3(vv_kappa)}** "
    f"and Gwet's AC1 **{fmt3(vv_ac1)}**."
)

md.append("\n## Operational three-way consensus\n")
md.append(
    f"- Unanimity between initial key + two validators: **{pct(three_unanim)}**."
)
md.append(
    f"- Three-way Fleiss' κ: **{fmt3(three_fleiss)}** "
    f"({ci_text('three_way_fleiss_kappa')})."
)
md.append(
    "- These are operational consensus statistics rather than reliability among "
    "three independent validators because the initial key is one of the three votes."
)

md.append("\n## Adjudication effects\n")
if not pd.isna(key_change):
    md.append(
        f"- Initial-key change rate among retained items: **{pct(key_change)}** "
        f"({ci_text('original_key_changed_rate_among_retained', percent=True)})."
    )
if not pd.isna(key_change_adj):
    md.append(
        f"- Initial-key change rate among adjudicated retained items: "
        f"**{pct(key_change_adj)}** "
        f"({ci_text('original_key_changed_rate_among_adjudicated_retained', percent=True)})."
    )
if not pd.isna(follow_maj):
    md.append(
        f"- Senior adjudication followed the provisional 2/3 majority in "
        f"**{pct(follow_maj)}** of evaluable adjudicated cases."
    )

md.append("\n## Interpretation note\n")
md.append(
    "All binary items are tacit and were assigned to two independent validators. "
    "Because validator pairs rotate across the nine-person pool, Krippendorff's nominal "
    "alpha is the primary global inter-rater reliability statistic. Cohen's kappa and "
    "Gwet's AC1 are complementary pooled diagnostics. With only two response categories "
    "(A/B), every valid non-unanimous three-vote configuration necessarily yields a 2/3 majority."
)

md.append("\n## Integrity checks\n")
for _, row in integrity.iterrows():
    md.append(f"- {row['check']}: {row['value']} (pass={row['pass']}).")

OUTPUT_MD.write_text("\n".join(md), encoding="utf-8")

# =============================================================================
# CONSOLE SUMMARY
# =============================================================================

print("=" * 80)
print("Binary-question validation statistics complete")
print(f"Pre-adjudication: {PRE_ADJUDICATION_FILE}")
print(f"Adjudicated:      {ADJUDICATED_FILE}")
print(f"Statistics:       {OUTPUT_XLSX}")
print(f"Markdown report:  {OUTPUT_MD}")
print(f"Bootstrap runs:   {N_BOOTSTRAP}")
print("=" * 80)
print(f"initial_binary_items: {N}")
print(f"adjudication_rate: {pct(adjud_rate)}")
print(f"adjudication_completion_rate: {pct(completion)}")
if not pd.isna(ret_rate):
    print(f"post_adjudication_retention_rate: {pct(ret_rate)}")
print(f"original_key_changed_rate_among_retained: {pct(key_change)}")
print(f"krippendorff_alpha_actual_validator_pool: {fmt3(alpha)}")
print(f"validator1_vs_validator2__raw_agreement: {pct(vv_raw)}")
print(f"validator1_vs_validator2__cohen_kappa: {fmt3(vv_kappa)}")
print(f"validator1_vs_validator2__gwet_ac1: {fmt3(vv_ac1)}")
print(f"three_way_unanimous_rate: {pct(three_unanim)}")
print(f"three_way_fleiss_kappa: {fmt3(three_fleiss)}")
print("=" * 80)

# Convenient notebook display
display(overall_metrics)
'''

path = Path("binary_validation_statistics_jupyter.py")
path.write_text(code, encoding="utf-8")

# Syntax check
compile(code, str(path), "exec")

print(f"Created and syntax-checked: {path}")


Created and syntax-checked: binary_validation_statistics_jupyter.py


In [3]:
%run binary_validation_statistics_jupyter.py

Binary-question validation statistics complete
Pre-adjudication: binary_questions_merged_not_yet_adjudicated.xlsx
Adjudicated:      binary_questions_adjudicated.xlsx
Statistics:       binary_validation_statistics.xlsx
Markdown report:  binary_validation_report.md
Bootstrap runs:   2000
initial_binary_items: 236
adjudication_rate: 13.6%
adjudication_completion_rate: 100.0%
post_adjudication_retention_rate: 97.9%
original_key_changed_rate_among_retained: 1.3%
krippendorff_alpha_actual_validator_pool: 0.881
validator1_vs_validator2__raw_agreement: 94.0%
validator1_vs_validator2__cohen_kappa: 0.881
validator1_vs_validator2__gwet_ac1: 0.881
three_way_unanimous_rate: 93.2%
three_way_fleiss_kappa: 0.909


,section,metric,value,n,ci95_low,ci95_high,note
0,dataset_flow,initial_binary_items,236.000000,236,NaN,NaN,
1,dataset_flow,automatic_accept_items,204.000000,236,NaN,NaN,
2,dataset_flow,adjudication_queue_items,32.000000,236,NaN,NaN,
3,dataset_flow,automatic_accept_rate,0.864407,236,0.814851,0.902289,
4,dataset_flow,adjudication_rate,0.135593,236,0.097711,0.185149,
5,adjudication,adjudication_completion_rate,1.000000,32,0.892821,1.000000,
6,adjudication,status_accepted_count,23.000000,32,NaN,NaN,
7,adjudication,status_accepted_rate_among_queued,0.718750,32,0.546255,0.844354,
8,adjudication,status_revised_count,4.000000,32,NaN,NaN,
9,adjudication,status_revised_rate_among_queued,0.125000,32,0.049701,0.280683,
